In [1]:
! pip install -q ftfy regex tqdm
! pip install -q git+https://github.com/openai/CLIP.git


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.2 MB/s eta 0:00:00


In [2]:
! pip install faiss-gpu-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 66.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you 

In [4]:
import os
import glob
import time
import torch
import clip
import numpy as np
from PIL import Image, UnidentifiedImageError
import kagglehub

# ================= CONFIG =================
BATCH_SIZES = [64, 128, 256, 512]
BACKBONES = [
    "RN50", "RN101", "RN50x4", "RN50x16", "RN50x64",
    "ViT-B/32", "ViT-B/16", "ViT-L/14", "ViT-L/14@336px"
]
N_IMAGES = 1000  # số ảnh test đầu tiên
DATASET_ID = "phucnguyenchau/keyframes-l06"
LOG_FILE = "clip_benchmark_fixed_log.txt"

WARMUP_ROUNDS = 2    # số vòng warm-up
BENCH_ROUNDS = 3     # số vòng đo, lấy trung bình
# ==========================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(">>> Using device:", device)


def get_base_dir_from_kaggle(dataset_id: str) -> str:
    root = kagglehub.dataset_download(dataset_id)
    candidates = [
        os.path.join(root, "version", "1", "keyframes"),
        os.path.join(root, "keyframes"),
        root,
    ]
    for c in candidates:
        if os.path.isdir(c) and glob.glob(os.path.join(c, "L06_V*")):
            return c
    matches = glob.glob(os.path.join(root, "**", "L06_V*"), recursive=True)
    return os.path.dirname(matches[0]) if matches else root


def collect_image_paths(base_dir: str, n=N_IMAGES):
    paths = sorted(glob.glob(os.path.join(base_dir, "**", "*.jpg"), recursive=True))
    return paths[:n]


def embed_batch(model, preprocess, image_paths, bs):
    # Load & preprocess images
    imgs = []
    for p in image_paths:
        try:
            imgs.append(preprocess(Image.open(p).convert("RGB")))
        except (FileNotFoundError, UnidentifiedImageError, OSError):
            continue

    imgs = torch.stack(imgs).to(device)
    n = len(imgs)

    # Warm-up
    with torch.no_grad():
        for _ in range(WARMUP_ROUNDS):
            for i in range(0, n, bs):
                model.encode_image(imgs[i:i + bs])

    # Benchmark rounds
    times = []
    for _ in range(BENCH_ROUNDS):
        torch.cuda.synchronize()
        start = time.time()
        with torch.no_grad():
            for i in range(0, n, bs):
                model.encode_image(imgs[i:i + bs])
        torch.cuda.synchronize()
        times.append(time.time() - start)

    avg_time = np.mean(times)
    return n / avg_time  # imgs/s


def benchmark_backbone(backbone, image_paths):
    results = []
    model, preprocess = clip.load(backbone, device=device)
    model.eval()

    for bs in BATCH_SIZES:
        try:
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(device)

            speed = embed_batch(model, preprocess, image_paths, bs)
            peak_vram = torch.cuda.max_memory_allocated(device) / 1e6

            # Latency test (text encoding)
            text = clip.tokenize(["a red car on the road"]).to(device)
            torch.cuda.synchronize()
            start = time.time()
            with torch.no_grad():
                for _ in range(100):
                    model.encode_text(text)
            torch.cuda.synchronize()
            latency = (time.time() - start) / 100 * 1000  # ms/query

            results.append((bs, speed, latency, peak_vram))
            print(f"[RESULT] {backbone}, batch={bs} | "
                  f"Speed={speed:.2f} imgs/s, Latency={latency:.2f} ms, VRAM={peak_vram:.2f} MB")

        except RuntimeError as e:
            results.append((bs, 0, 0, -1))
            print(f"[WARN] {backbone}, batch={bs} OOM: {e}")
    return results


if __name__ == "__main__":
    base_dir = get_base_dir_from_kaggle(DATASET_ID)
    image_paths = collect_image_paths(base_dir)
    print(f"[INFO] Using {len(image_paths)} images for benchmark.")

    with open(LOG_FILE, "w") as f:
        f.write("# CLIP Benchmark Log (Fixed)\n\n")
        for backbone in BACKBONES:
            print(f"\n[INFO] Benchmarking backbone: {backbone}")
            f.write(f"## {backbone}\n")
            results = benchmark_backbone(backbone, image_paths)
            for bs, s, l, v in results:
                line = f"Batch={bs}: Speed={s:.2f} imgs/s, Latency={l:.2f} ms, VRAM={v:.2f} MB"
                f.write(line + "\n")
            f.write("\n")

    print(f"[INFO] Benchmark log saved -> {LOG_FILE}")


>>> Using device: cuda
[INFO] Using 1000 images for benchmark.

[INFO] Benchmarking backbone: RN50
[RESULT] RN50, batch=64 | Speed=655.59 imgs/s, Latency=9.28 ms, VRAM=2266.97 MB
[RESULT] RN50, batch=128 | Speed=638.86 imgs/s, Latency=8.48 ms, VRAM=2646.04 MB
[RESULT] RN50, batch=256 | Speed=636.32 imgs/s, Latency=8.50 ms, VRAM=3403.90 MB
[RESULT] RN50, batch=512 | Speed=647.14 imgs/s, Latency=8.46 ms, VRAM=4920.67 MB

[INFO] Benchmarking backbone: RN101
[RESULT] RN101, batch=64 | Speed=402.53 imgs/s, Latency=8.45 ms, VRAM=2302.14 MB
[RESULT] RN101, batch=128 | Speed=406.60 imgs/s, Latency=10.06 ms, VRAM=2681.22 MB
[RESULT] RN101, batch=256 | Speed=414.46 imgs/s, Latency=8.31 ms, VRAM=3439.08 MB
[RESULT] RN101, batch=512 | Speed=400.92 imgs/s, Latency=8.40 ms, VRAM=4955.84 MB

[INFO] Benchmarking backbone: RN50x4
[RESULT] RN50x4, batch=64 | Speed=171.07 imgs/s, Latency=10.32 ms, VRAM=3277.94 MB
[RESULT] RN50x4, batch=128 | Speed=173.56 imgs/s, Latency=12.12 ms, VRAM=4001.22 MB
[RESULT]